# Exploratory analysis

What do GB half-hourly prices look like? Four views: monthly baseload and peak, average daily shape by season, negative prices, and volatility by time of day. Every chart is saved to `figures/` for the README.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
from gbpricing.features import add_time_features
from gbpricing.holidays import load_bank_holidays

hols = load_bank_holidays(DATA / "raw/bank_holidays.json")
prices = add_time_features(pd.read_parquet(DATA / "processed/mid_clean.parquet"), hols)
print(f"{len(prices):,} half-hours, {prices['settlement_date'].min():%d %b %Y} to {prices['settlement_date'].max():%d %b %Y}")

## Monthly baseload and peak

Baseload = average of all half-hours in the month. Peak = average of weekday 07:00-19:00 (local). The gap between them is what the peak product prices.

In [ ]:
from gbpricing.eda import monthly_base_peak

mbp = monthly_base_peak(prices)
ax = mbp[["base_gbp_mwh", "peak_gbp_mwh"]].plot(title="Monthly baseload and peak (GBP/MWh)")
ax.set_xlabel(""); ax.set_ylabel("GBP/MWh")
plt.savefig(FIG / "02_monthly_base_peak.png", dpi=150, bbox_inches="tight")
mbp.round(2).tail(12)

## Average daily shape by season

Each line is the average price at each local half-hour. Look for the evening peak (around 17:00-19:00), the overnight trough, and in summer a midday dip as solar output floods the market.

In [ ]:
from gbpricing.eda import daily_shape_by_season

shape = daily_shape_by_season(prices)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, dt_ in zip(axes, ["weekday", "weekend"]):
    for season in ["Winter", "Spring", "Summer", "Autumn"]:
        ax.plot(shape.index / 2, shape[(season, dt_)], label=season)
    ax.set_title(f"{dt_.title()}s"); ax.set_xlabel("Local hour"); ax.set_xticks(range(0, 25, 3))
axes[0].set_ylabel("GBP/MWh"); axes[0].legend()
plt.savefig(FIG / "02_daily_shape.png", dpi=150, bbox_inches="tight")

## Negative prices

How often prices go below zero, by month. Negative prices happen when inflexible or subsidised generation (mostly wind and solar) exceeds demand.

In [ ]:
from gbpricing.eda import negative_price_stats

neg = negative_price_stats(prices)
share = (neg["negative_share"] * 100).rename(lambda d: d.strftime("%b %y"))
ax = share.plot(kind="bar", title="Share of half-hours with negative prices (%)")
ax.set_xlabel("")
plt.savefig(FIG / "02_negative_prices.png", dpi=150, bbox_inches="tight")
print(f"Total negative half-hours: {neg['negative_periods'].sum():,}  |  lowest price: {neg['min_gbp_mwh'].min():.2f}")

## Volatility by time of day

Two measures per local half-hour: the spread around the monthly level, and the spread of day-on-day changes. Peaks in the evening mean that's where the price risk sits.

In [ ]:
from gbpricing.eda import volatility_by_slot

vol = volatility_by_slot(prices)
ax = vol.set_index(vol.index / 2).plot(title="Price volatility by local half-hour (GBP/MWh)")
ax.set_xlabel("Local hour"); ax.set_xticks(range(0, 25, 3))
plt.savefig(FIG / "02_volatility.png", dpi=150, bbox_inches="tight")

## What I found
